1 — Cài thư viện

In [ ]:
# !pip install -q sentence-transformers scikit-learn pandas numpy joblib

2 — Upload CSV

In [ ]:
# from google.colab import files

# uploaded = files.upload()

3 — Đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np

input_path = "reviews_for_colab_embedding_e5_aspect.csv"

df = pd.read_csv(input_path, engine='python', on_bad_lines='skip')

print("Shape:", df.shape)
print("Missing embedding_text:", df["embedding_text"].isna().sum())
print("Missing positive_clean:", df["positive_clean"].isna().sum())
print("Missing negative_clean:", df["negative_clean"].isna().sum())

df.head()

Shape: (18630, 11)
Missing embedding_text: 0
Missing positive_clean: 972
Missing negative_clean: 3314


,review_id,hotel_id,hotel_name,province_id,province_name,rating,sentiment_label,language,embedding_text,positive_clean,negative_clean
0,dcfcffe3a1b6245f08a6fe4cefc4c66e,70e7270e63138fa24635a7d19dc7ace5,Homestay Khoa Chaudoc,an-giang,An Giang,10.0,positive,en,"POSITIVE: Firstly, I want to say what a lovely...","Firstly, I want to say what a lovely warm welc...",Nothing to dislike
1,f0530b3fc2a36e4a89d7cea83b88d101,70e7270e63138fa24635a7d19dc7ace5,Homestay Khoa Chaudoc,an-giang,An Giang,9.0,positive,en,"POSITIVE: Great location near the market, temp...","Great location near the market, temples and ri...",Strange recommendations from Vi for places for...
2,ebc706be461bd85030634571492ba861,70e7270e63138fa24635a7d19dc7ace5,Homestay Khoa Chaudoc,an-giang,An Giang,10.0,positive,en,"POSITIVE: Extremely central location, clean fa...","Extremely central location, clean facilities, ...",No elevator which is fine for most people but ...
3,cc905459bfee8a357867d701d9414c84,70e7270e63138fa24635a7d19dc7ace5,Homestay Khoa Chaudoc,an-giang,An Giang,10.0,positive,fr,"POSITIVE: L'accueil de Vi, de sa maman et de s...","L'accueil de Vi, de sa maman et de son papa so...",Rien !
4,a2d8a867d8ef21e1e812f509708f6610,70e7270e63138fa24635a7d19dc7ace5,Homestay Khoa Chaudoc,an-giang,An Giang,7.0,neutral,de,POSITIVE: Wir hatten ein Zimmer mit drei große...,"Wir hatten ein Zimmer mit drei großen, einfach...",NaN


4 — Cấu hình

In [ ]:
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-large"

BATCH_SIZE_EMBEDDING = 32
BATCH_SIZE_ASPECT = 32

N_REVIEW_TOPICS = 12
PCA_COMPONENTS = 50
RANDOM_STATE = 42

ASPECT_SIMILARITY_THRESHOLD = 0.35
MAX_ASPECTS_PER_TEXT = 3

In [ ]:
BATCH_SIZE_EMBEDDING = 16
BATCH_SIZE_ASPECT = 16

5 — Load model E5-Large

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device
)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

6 — Tạo review embedding bằng E5-Large

In [ ]:
# E5 cần prefix. Review là văn bản/corpus nên dùng "passage: "
texts = [
    "passage: " + str(text)
    for text in df["embedding_text"].fillna("").astype(str).tolist()
]

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE_EMBEDDING,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1165 [00:00<?, ?it/s]

Embedding shape: (18630, 1024)


7 — PCA cho topic review

In [ ]:
from sklearn.decomposition import PCA

n_samples, n_features = embeddings.shape
n_components = min(PCA_COMPONENTS, n_features, n_samples - 1)

pca = PCA(
    n_components=n_components,
    random_state=RANDOM_STATE
)

X_pca = pca.fit_transform(embeddings)

explained_variance = float(np.sum(pca.explained_variance_ratio_))

print("X_pca shape:", X_pca.shape)
print("PCA explained variance:", explained_variance)

X_pca shape: (18630, 50)
PCA explained variance: 0.5988242030143738


8 — MiniBatchKMeans topic

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

kmeans = MiniBatchKMeans(
    n_clusters=N_REVIEW_TOPICS,
    random_state=RANDOM_STATE,
    batch_size=1024,
    n_init=10,
    max_iter=300
)

labels = kmeans.fit_predict(X_pca)

df["review_topic"] = labels.astype(int)
df["topic_model"] = "Multilingual-E5-Large + PCA + MiniBatchKMeans"
df["topic_n_clusters"] = N_REVIEW_TOPICS

df["review_topic"].value_counts().sort_index()

,count
review_topic,
0,1000
1,1279
2,2058
3,1593
4,2374
5,3277
6,955
7,418
8,577


9 — Metrics topic

In [ ]:
if len(set(labels)) > 1:
    silhouette = silhouette_score(X_pca, labels)
    davies_bouldin = davies_bouldin_score(X_pca, labels)
    calinski_harabasz = calinski_harabasz_score(X_pca, labels)
else:
    silhouette = None
    davies_bouldin = None
    calinski_harabasz = None

metrics = {
    "model": "Multilingual-E5-Large + PCA + MiniBatchKMeans",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "n_topics": N_REVIEW_TOPICS,
    "pca_components": n_components,
    "pca_explained_variance": explained_variance,
    "silhouette_score": silhouette,
    "davies_bouldin_index": davies_bouldin,
    "calinski_harabasz_index": calinski_harabasz,
}

metrics

{'model': 'Multilingual-E5-Large + PCA + MiniBatchKMeans',
 'embedding_model': 'intfloat/multilingual-e5-large',
 'n_topics': 12,
 'pca_components': 50,
 'pca_explained_variance': 0.5988242030143738,
 'silhouette_score': np.float32(0.099168755),
 'davies_bouldin_index': np.float64(2.3996047650406926),
 'calinski_harabasz_index': np.float32(1101.098)}

10 — Aspect prompts

In [ ]:
ASPECT_PROMPTS = {
    "room": (
        "hotel room, bed, bathroom, toilet, shower, balcony, room size, furniture, air conditioner; "
        "phòng khách sạn, giường, nhà vệ sinh, phòng tắm, vòi sen, ban công, diện tích phòng, nội thất, điều hòa"
    ),

    "cleanliness": (
        "cleanliness, hygiene, dirty room, smell, dust, mold, bathroom cleanliness; "
        "sạch sẽ, vệ sinh, phòng bẩn, mùi hôi, bụi, ẩm mốc, nhà tắm bẩn"
    ),

    "service": (
        "hotel staff, receptionist, host, owner, service, friendly, helpful, support, hospitality; "
        "nhân viên, lễ tân, chủ nhà, chủ khách sạn, dịch vụ, thân thiện, nhiệt tình, hỗ trợ, hiếu khách"
    ),

    "location": (
        "hotel location, city center, beach, airport, station, bus, market, nearby attractions, transportation; "
        "vị trí, trung tâm, gần biển, sân bay, nhà ga, bến xe, chợ, điểm tham quan, giao thông thuận tiện"
    ),

    "price": (
        "price, value for money, expensive, cheap, cost, worth the money, reasonable price; "
        "giá cả, đáng tiền, đắt, rẻ, chi phí, giá hợp lý, xứng đáng với số tiền"
    ),

    "food": (
        "breakfast, food, restaurant, coffee, tea, meal, dinner, drinks; "
        "bữa sáng, đồ ăn, nhà hàng, cà phê, trà, bữa ăn, bữa tối, đồ uống"
    ),

    "facilities": (
        "hotel facilities, swimming pool, parking, elevator, gym, laundry, motorbike rental, amenities; "
        "tiện nghi, hồ bơi, bãi đỗ xe, thang máy, phòng gym, giặt ủi, thuê xe máy, trang thiết bị"
    ),

    "wifi": (
        "wifi, wi-fi, internet, network connection, slow internet, weak wifi; "
        "wifi, internet, kết nối mạng, mạng yếu, wifi chậm"
    ),

    "noise": (
        "noise, noisy, loud music, soundproofing, quiet room, street noise; "
        "tiếng ồn, ồn ào, nhạc lớn, cách âm, phòng yên tĩnh, tiếng ồn ngoài đường"
    ),

    "checkin": (
        "check in, check out, booking, reservation, arrival, late check-in, early check-in; "
        "nhận phòng, trả phòng, đặt phòng, đặt chỗ, đến nơi, nhận phòng muộn, nhận phòng sớm"
    )
}

11 — Encode aspect prompts

In [ ]:
aspect_names = list(ASPECT_PROMPTS.keys())

# E5: prompt aspect là query/mô tả khái niệm
aspect_texts = [
    "query: " + ASPECT_PROMPTS[name]
    for name in aspect_names
]

aspect_embeddings = model.encode(
    aspect_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Aspect embeddings shape:", aspect_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Aspect embeddings shape: (10, 1024)


12 — Hàm gán aspect

In [ ]:
def is_valid_aspect_text(text):
    if text is None:
        return False

    text = str(text).strip()

    if text == "":
        return False

    if len(text) < 10:
        return False

    return True


def select_aspects_by_similarity(text_embedding, aspect_names, aspect_embeddings):
    similarities = np.dot(aspect_embeddings, text_embedding)

    ranked_indices = np.argsort(similarities)[::-1]

    selected_aspects = []
    selected_scores = {}

    for idx in ranked_indices:
        aspect = aspect_names[idx]
        score = float(similarities[idx])

        if score < ASPECT_SIMILARITY_THRESHOLD:
            continue

        selected_aspects.append(aspect)
        selected_scores[aspect] = round(score, 4)

        if len(selected_aspects) >= MAX_ASPECTS_PER_TEXT:
            break

    return selected_aspects, selected_scores

13 — Chuẩn bị positive/negative text

In [ ]:
positive_items = []
negative_items = []

for idx, row in df.iterrows():
    review_id = str(row["review_id"])

    pos_text = row.get("positive_clean", None)
    neg_text = row.get("negative_clean", None)

    if is_valid_aspect_text(pos_text):
        positive_items.append((idx, review_id, str(pos_text)))

    if is_valid_aspect_text(neg_text):
        negative_items.append((idx, review_id, str(neg_text)))

print("Positive texts:", len(positive_items))
print("Negative texts:", len(negative_items))

Positive texts: 17124
Negative texts: 14627


14 — Encode positive/negative text

In [ ]:
positive_texts = [
    "passage: " + text
    for _, _, text in positive_items
]

negative_texts = [
    "passage: " + text
    for _, _, text in negative_items
]

positive_aspect_embeddings = model.encode(
    positive_texts,
    batch_size=BATCH_SIZE_ASPECT,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

negative_aspect_embeddings = model.encode(
    negative_texts,
    batch_size=BATCH_SIZE_ASPECT,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Positive aspect embeddings:", positive_aspect_embeddings.shape)
print("Negative aspect embeddings:", negative_aspect_embeddings.shape)

Batches:   0%|          | 0/1071 [00:00<?, ?it/s]

Batches:   0%|          | 0/915 [00:00<?, ?it/s]

Positive aspect embeddings: (17124, 1024)
Negative aspect embeddings: (14627, 1024)


15 — Gán aspect

In [ ]:
positive_result = {}
negative_result = {}

for (idx, review_id, _), emb in zip(positive_items, positive_aspect_embeddings):
    aspects, scores = select_aspects_by_similarity(
        emb,
        aspect_names,
        aspect_embeddings
    )

    positive_result[review_id] = {
        "aspects": aspects,
        "scores": scores
    }

for (idx, review_id, _), emb in zip(negative_items, negative_aspect_embeddings):
    aspects, scores = select_aspects_by_similarity(
        emb,
        aspect_names,
        aspect_embeddings
    )

    negative_result[review_id] = {
        "aspects": aspects,
        "scores": scores
    }


positive_aspects_col = []
negative_aspects_col = []
review_aspects_col = []
positive_scores_col = []
negative_scores_col = []

for _, row in df.iterrows():
    review_id = str(row["review_id"])

    pos_data = positive_result.get(review_id, {"aspects": [], "scores": {}})
    neg_data = negative_result.get(review_id, {"aspects": [], "scores": {}})

    pos_aspects = pos_data["aspects"]
    neg_aspects = neg_data["aspects"]

    review_aspects = sorted(list(set(pos_aspects + neg_aspects)))

    positive_aspects_col.append(pos_aspects)
    negative_aspects_col.append(neg_aspects)
    review_aspects_col.append(review_aspects)
    positive_scores_col.append(pos_data["scores"])
    negative_scores_col.append(neg_data["scores"])


df["positive_aspects"] = positive_aspects_col
df["negative_aspects"] = negative_aspects_col
df["review_aspects"] = review_aspects_col
df["positive_aspect_scores"] = positive_scores_col
df["negative_aspect_scores"] = negative_scores_col

df["aspect_method"] = "multilingual_e5_large_embedding_similarity"
df["aspect_model"] = EMBEDDING_MODEL_NAME
df["aspect_similarity_threshold"] = ASPECT_SIMILARITY_THRESHOLD

df[
    [
        "review_id",
        "positive_aspects",
        "negative_aspects",
        "review_aspects",
        "positive_aspect_scores",
        "negative_aspect_scores"
    ]
].head()

,review_id,positive_aspects,negative_aspects,review_aspects,positive_aspect_scores,negative_aspect_scores
0,dcfcffe3a1b6245f08a6fe4cefc4c66e,"[food, price, location]","[cleanliness, noise, price]","[cleanliness, food, location, noise, price]","{'food': 0.8123, 'price': 0.8111, 'location': ...","{'cleanliness': 0.8129, 'noise': 0.8081, 'pric..."
1,f0530b3fc2a36e4a89d7cea83b88d101,"[location, food, service]","[noise, food, location]","[food, location, noise, service]","{'location': 0.838, 'food': 0.8337, 'service':...","{'noise': 0.8282, 'food': 0.8138, 'location': ..."
2,ebc706be461bd85030634571492ba861,"[food, location, facilities]","[room, cleanliness, facilities]","[cleanliness, facilities, food, location, room]","{'food': 0.83, 'location': 0.8286, 'facilities...","{'room': 0.8211, 'cleanliness': 0.8104, 'facil..."
3,cc905459bfee8a357867d701d9414c84,"[location, service, facilities]",[],"[facilities, location, service]","{'location': 0.8133, 'service': 0.8083, 'facil...",{}
4,a2d8a867d8ef21e1e812f509708f6610,"[room, service, food]",[],"[food, room, service]","{'room': 0.8175, 'service': 0.8119, 'food': 0....",{}


16 — Xuất topic counts

In [ ]:
topic_counts = (
    df.groupby("review_topic")
    .agg(num_reviews=("review_id", "count"))
    .reset_index()
    .sort_values("review_topic")
)

topic_counts.to_csv(
    "review_topic_counts_e5_large.csv",
    index=False,
    encoding="utf-8-sig"
)

topic_counts

,review_topic,num_reviews
0,0,1000
1,1,1279
2,2,2058
3,3,1593
4,4,2374
5,5,3277
6,6,955
7,7,418
8,8,577
9,9,767


17 — Xuất topic samples

In [ ]:
samples_per_topic = 30

sample_rows = []

for topic in sorted(df["review_topic"].unique()):
    topic_df = df[df["review_topic"] == topic].head(samples_per_topic)
    sample_rows.append(topic_df)

topic_samples = pd.concat(sample_rows, ignore_index=True)

topic_samples_export = topic_samples[
    [
        "review_id",
        "review_topic",
        "hotel_name",
        "province_id",
        "rating",
        "sentiment_label",
        "language",
        "embedding_text"
    ]
]

topic_samples_export.to_csv(
    "review_topic_samples_e5_large.csv",
    index=False,
    encoding="utf-8-sig"
)

topic_samples_export.head()

,review_id,review_topic,hotel_name,province_id,rating,sentiment_label,language,embedding_text
0,01314b4c7c30443820f7dc1024f4c062,0,HaiVan HosTel,an-giang,7.0,neutral,it,POSITIVE: Molto a buon mercato [SEP] NEGATIVE:...
1,9fafba40aa3a9f3a1a85a63340955dec,0,FaMi Nguyễn Homestay Châu Đốc,an-giang,5.0,negative,de,"POSITIVE: Sauber, [SEP] NEGATIVE: Die 4 Häusch..."
2,8a17b71bc2ee53f2af3eca60749163d9,0,Sang Như Ngọc Resort,an-giang,3.0,negative,da,"POSITIVE: Bjergene, poolen. [SEP] NEGATIVE: De..."
3,69e687291911661e6e077f19ad277df9,0,Hoang Duc Hotel,an-giang,6.0,neutral,en,POSITIVE: Big nice and clean [SEP] NEGATIVE: I...
4,a0983162fa0fae0c969285c049235b40,0,hotel đăng khôi 2,an-giang,7.0,neutral,vi,POSITIVE: Trong khu dân cư khá yên tĩnh [SEP] ...


18 — Lưu metrics

In [ ]:
metrics_df = pd.DataFrame([metrics])

metrics_df.to_csv(
    "review_topic_clustering_metrics_e5_large.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics_df

,model,embedding_model,n_topics,pca_components,pca_explained_variance,silhouette_score,davies_bouldin_index,calinski_harabasz_index
0,Multilingual-E5-Large + PCA + MiniBatchKMeans,intfloat/multilingual-e5-large,12,50,0.598824,0.099169,2.399605,1101.098022


19 — Lưu file pkl chính

In [ ]:
df["embedding"] = [vec.astype(float).tolist() for vec in embeddings]
df["embedding_model"] = EMBEDDING_MODEL_NAME
df["embedding_dim"] = embeddings.shape[1]

output_df = df[
    [
        "review_id",

        "embedding",
        "embedding_model",
        "embedding_dim",

        "review_topic",
        "topic_model",
        "topic_n_clusters",

        "positive_aspects",
        "negative_aspects",
        "review_aspects",
        "positive_aspect_scores",
        "negative_aspect_scores",
        "aspect_method",
        "aspect_model",
        "aspect_similarity_threshold",
    ]
].copy()

output_df["review_id"] = output_df["review_id"].astype(str)
output_df["embedding_model"] = output_df["embedding_model"].astype(str)
output_df["embedding_dim"] = output_df["embedding_dim"].astype(int)
output_df["review_topic"] = output_df["review_topic"].astype(int)
output_df["topic_model"] = output_df["topic_model"].astype(str)
output_df["topic_n_clusters"] = output_df["topic_n_clusters"].astype(int)
output_df["aspect_method"] = output_df["aspect_method"].astype(str)
output_df["aspect_model"] = output_df["aspect_model"].astype(str)
output_df["aspect_similarity_threshold"] = output_df["aspect_similarity_threshold"].astype(float)

output_df.to_pickle(
    "colab_review_embeddings_topics_aspects_e5_large.pkl",
    protocol=4
)

print("Saved colab_review_embeddings_topics_aspects_e5_large.pkl")
print(output_df.shape)

Saved colab_review_embeddings_topics_aspects_e5_large.pkl
(18630, 15)


20 — Lưu PCA + KMeans

In [ ]:
import joblib

joblib.dump(pca, "review_topic_pca_e5_large.pkl")
joblib.dump(kmeans, "review_topic_kmeans_e5_large.pkl")

print("Saved PCA and KMeans models.")

Saved PCA and KMeans models.


21 — Download kết quả

In [ ]:
from google.colab import files

files.download("colab_review_embeddings_topics_aspects_e5_large.pkl")
files.download("review_topic_counts_e5_large.csv")
files.download("review_topic_samples_e5_large.csv")
files.download("review_topic_clustering_metrics_e5_large.csv")
files.download("review_topic_pca_e5_large.pkl")
files.download("review_topic_kmeans_e5_large.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>